# Núcleo 3 — Simulador de sensibilidad de la coparticipación

**La pregunta.** Si la recaudación de los impuestos nacionales que se coparticipan cae
(o sube) un determinado porcentaje, ¿cuánta plata pierde (o gana) Córdoba?

Como la coparticipación es automática y se reparte con coeficientes fijos, la respuesta
es una cuenta: el cambio en la recaudación, multiplicado por el coeficiente de cada
provincia. El simulador (`scripts/simulator.py`) hace esa cuenta para las 24
jurisdicciones, con los coeficientes de la Ley 23.548 del Núcleo 2.

**Qué no es.** No es un pronóstico: no dice cuánto va a caer la recaudación, solo qué
pasa si cae tanto. Tampoco modela cómo reaccionaría cada provincia (más impuestos
propios, menos gasto, deuda) ni las transferencias no automáticas, que Nación decide
caso por caso.

El simulador parte de tres supuestos (el detalle completo está en el docstring de
`scripts/simulator.py` y en `docs/methodology.md`): `recaudacion_base` es la masa
coparticipable TOTAL (incluye lo que retiene la Nación y el Fondo ATN, no solo lo que
reciben las provincias); uso únicamente los coeficientes de Ley 23.548, sin la
compensación del Consenso Fiscal (Ley 27.429), porque es un monto históricamente fijo
en pesos, no una proporción de la recaudación corriente; y los adelantos de
coparticipación (por ejemplo, el Decreto 219/2026) son una partida aparte —anticipos a
cuenta— que no cambian el coeficiente de reparto de nadie, así que no los modelo
acá.

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

BASE_DIR = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
sys.path.insert(0, str(BASE_DIR / "scripts"))

from simulator import simular_shock
from coeficientes_ley23548 import COEFICIENTES_COPARTICIPACION

COLOR_NEGATIVO = "#e34948"
COLOR_POSITIVO = "#2a78d6"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID_COLOR = "#e1e0d9"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.edgecolor": GRID_COLOR,
    "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "text.color": INK_PRIMARY,
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
})

FIGURES_DIR = BASE_DIR / "output" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

suma_coeficientes_provincias = sum(COEFICIENTES_COPARTICIPACION.values())
print(f"Suma de los 24 coeficientes (participación de las provincias en la masa coparticipable total): {suma_coeficientes_provincias:.4f}")

Suma de los 24 coeficientes (participación de las provincias en la masa coparticipable total): 0.5876


## 1. Calibración con una cifra pública: caída de recaudación de comienzos de 2026

IARAF estimó que en el primer cuatrimestre de 2026 la recaudación tributaria nacional
cayó un 6,7% real interanual, unos $5,1 billones
([informe del 5 de mayo de 2026](https://www.iaraf.org/index.php/13-areas-de-estudio/area-fiscal/723-informe-economico-2026-05-05)).
De esa caída, $1,4 billones habrían sido menos coparticipación para las provincias. Esta
segunda cifra no figura en el resumen que IARAF publica en su web: estaría en el informe
completo, que está en Google Drive y no pude abrir para verificarla. Más abajo muestro
cuánto cambia el resultado si fuera $1,3 billones, la cifra que publicó Ámbito.

No encontré una cifra oficial de "masa coparticipable de referencia" para ese período,
así que la estimo en `scripts/calibracion_iaraf.py`: parto de lo que recibieron las 24
jurisdicciones en 2025 por la Ley 23.548 (Núcleo 1, sin la compensación del Consenso
Fiscal, que no sigue los coeficientes), tomo 4/12 del año, lo llevo a precios de
enero-abril de 2026 con el IPC y lo divido por la suma de los coeficientes para llegar
a la masa total. Después busco la variación % que reproduce los $1,4 billones.

In [2]:
from calibracion_iaraf import IMPACTO_PROVINCIAS_IARAF, calibrar

calibracion = calibrar()
recaudacion_base_4m = calibracion["recaudacion_base"]
variacion_implicita = calibracion["variacion_implicita"]

print(f"Recibido por las 24 jurisdicciones en 2025 (sin compensación): ${calibracion['recibido_anio_base']:,.0f}")
print(f"Ajuste de precios 2025 -> ene-abr 2026 (IPC):                  x{calibracion['ajuste_precios']:.4f}")
print(f"Recibido estimado en el cuatrimestre, a precios de 2026:        ${calibracion['recibido_cuatrimestre']:,.0f}")
print(f"Masa coparticipable total estimada (4 meses):                   ${recaudacion_base_4m:,.0f}")

Recibido por las 24 jurisdicciones en 2025 (sin compensación): $54,748,974,222,900
Ajuste de precios 2025 -> ene-abr 2026 (IPC):                  x1.2148
Recibido estimado en el cuatrimestre, a precios de 2026:        $22,169,385,009,496
Masa coparticipable total estimada (4 meses):                   $37,728,701,513,778


In [3]:
print(f"Variación % de recaudación implícita: {variacion_implicita:.2%}")

resultado_calibracion = simular_shock(variacion_implicita, recaudacion_base_4m, COEFICIENTES_COPARTICIPACION)
print(f"Impacto total en provincias (modelo): ${resultado_calibracion['total_impacto_provincias']:,.0f}")
print(f"Impacto reportado por IARAF:          ${IMPACTO_PROVINCIAS_IARAF:,.0f}")

Variación % de recaudación implícita: -6.32%
Impacto total en provincias (modelo): $-1,400,000,000,000
Impacto reportado por IARAF:          $-1,400,000,000,000


Por construcción, el modelo reproduce exactamente el impacto de \$1,4 billones (de ahí
sale la variación % implícita): esto es una calibración, no una validación ni una
predicción. Dos aclaraciones sobre qué depende de qué:

- El impacto en pesos sobre cada provincia no depende de la masa que estimé: son los
  \$1,4 billones repartidos según los coeficientes. Lo que sí depende de esa estimación
  es la variación implícita, ~6,3%. En la sección 3 la contrasto con lo que Córdoba
  efectivamente cobró en el cuatrimestre.
- El ajuste de precios importa. Si dejara la masa en pesos promedio de 2025 contra un
  impacto medido en pesos de comienzos de 2026 (21,5% más caros según el IPC), la
  variación implícita saldría en ~7,7%, sobreestimada en esa misma proporción.
- Como el modelo es lineal, todo escala con la cifra de impacto. Si en vez de \$1,4
  billones la pérdida de las provincias fuera de \$1,3 billones (la cifra que publicó
  Ámbito el 5 de mayo de 2026, a partir de datos que difundió el diputado Guillermo
  Michel), la variación implícita sería ~5,9% y el impacto sobre Córdoba de unos
  \$111.500 millones en vez de \$120.100 millones. Las dos cifras no miden
  necesariamente lo mismo (transferencias totales o solo coparticipación).

En una versión anterior presentaba como chequeo que la masa afectada (~46,7% de la
caída total de \$5,1 billones) estaba en el rango habitual de la parte coparticipable de
la recaudación. No es un chequeo: ese 46,7% sale de \$1,4 billones / 0,5876 / \$5,1
billones y da igual con cualquier masa estimada. Solo dice que las dos cifras de IARAF
son coherentes entre sí.

## 2. Impacto por provincia

¿Cómo se reparte ese impacto de \$1,4 billones entre las 24 jurisdicciones? La tabla
muestra las tres que más pierden en pesos, la que menos pierde y el total.

In [4]:
resultado_escenario = simular_shock(variacion_implicita, recaudacion_base_4m, COEFICIENTES_COPARTICIPACION)
impacto_df = pd.DataFrame(resultado_escenario["por_provincia"]).set_index("provincia")
impacto_df["corresponde_sin_shock"] = impacto_df["coeficiente"] * recaudacion_base_4m

cordoba_impacto = impacto_df.loc["Córdoba", "impacto_pesos"]
print(f"Impacto estimado para Córdoba: ${cordoba_impacto / 1e9:,.1f} mil millones en el cuatrimestre")

filas = ["Buenos Aires", "Santa Fe", "Córdoba", "Tierra del Fuego"]
tabla = impacto_df.loc[filas, ["impacto_pesos", "corresponde_sin_shock"]]
tabla.loc["Total de las 24"] = impacto_df[["impacto_pesos", "corresponde_sin_shock"]].sum()
tabla["% de lo que le corresponde"] = tabla["impacto_pesos"] / tabla["corresponde_sin_shock"]
tabla = tabla.drop(columns="corresponde_sin_shock").rename(columns={"impacto_pesos": "Impacto (miles de millones de $ de ene-abr 2026)"})
tabla.style.format({
    "Impacto (miles de millones de $ de ene-abr 2026)": lambda v: f"{v / 1e9:,.1f}".replace(",", "X").replace(".", ",").replace("X", "."),
    "% de lo que le corresponde": lambda v: f"{v:.1%}".replace(".", ","),
})

Impacto estimado para Córdoba: $-120.1 mil millones en el cuatrimestre


,Impacto (miles de millones de $ de ene-abr 2026),% de lo que le corresponde
provincia,,
Buenos Aires,"-297,0","-6,3%"
Santa Fe,"-120,9","-6,3%"
Córdoba,"-120,1","-6,3%"
Tierra del Fuego,"-16,7","-6,3%"
Total de las 24,"-1.400,0","-6,3%"


Córdoba perdería unos \$120.100 millones de coparticipación en el cuatrimestre, casi lo
mismo que Santa Fe y menos de la mitad que Buenos Aires. Las diferencias en pesos solo
reflejan el tamaño de cada coeficiente.

En términos relativos, la última columna lo muestra: como el simulador reparte el shock
en proporción al coeficiente de cada provincia, todas pierden exactamente el mismo
porcentaje de lo que les corresponde por coeficiente, un 6,3%, igual que la masa. El Núcleo 5 muestra que el supuesto
es razonable: sin la compensación del Consenso Fiscal, lo que cada jurisdicción recibió
en 2025 coincide con su coeficiente legal con diferencias menores a 0,2 puntos.

Bajo este supuesto de reparto proporcional, ninguna provincia está más expuesta que otra
a un shock general de recaudación en términos de su coparticipación. Lo que sí cambia la
gravedad del golpe es cuánto depende cada provincia de la coparticipación en su propio
presupuesto, que es lo que mido para Córdoba en el Núcleo 4.

## 3. Contraste con lo que efectivamente cobró Córdoba

La calibración no se puede validar con la masa coparticipable de 2026, pero sí con
Córdoba. El archivo de Rentas del Núcleo 4 trae lo que la provincia cobró mes a mes
hasta agosto de 2026. Comparo enero-abril de 2026 con enero-abril de 2025, cada mes
llevado a precios del mismo mes de 2026 con el IPC, y uso solo la parte de la
coparticipación que sigue los coeficientes (sin la compensación ni el bono del Consenso
Fiscal), que es lo que mueve el simulador.

In [5]:
from calibracion_iaraf import contrastar_con_cordoba

contraste = contrastar_con_cordoba()
print(f"Variación real observada (Rentas de Córdoba): {contraste['variacion_real_observada']:.1%}")
print(f"Variación del modelo (calibrado con IARAF):   {contraste['variacion_modelo']:.1%}")
print(f"Pérdida observada:  ${contraste['perdida_observada_pesos'] / 1e9:,.1f} mil millones")
print(f"Pérdida del modelo: ${contraste['perdida_modelo_pesos'] / 1e9:,.1f} mil millones")
print(f"El modelo subestima la pérdida en {1 - contraste['perdida_modelo_pesos'] / contraste['perdida_observada_pesos']:.0%}")

Variación real observada (Rentas de Córdoba): -7.3%
Variación del modelo (calibrado con IARAF):   -6.3%
Pérdida observada:  $-134.1 mil millones
Pérdida del modelo: $-120.1 mil millones
El modelo subestima la pérdida en 10%


Córdoba perdió en el cuatrimestre un 7,3% real de su coparticipación proporcional, unos
\$134.100 millones a precios de 2026. El modelo calibrado con IARAF estima 6,3% y
\$120.100 millones: el orden de magnitud es el correcto, pero subestima la pérdida en
alrededor de 10%.

Si todas las provincias hubieran perdido lo mismo que Córdoba (en el modelo pierden el
mismo porcentaje), la pérdida total habría sido de unos \$1,56 billones, más que los
\$1,4 billones de la calibración y que los \$1,3 billones que publicó Ámbito. No puedo
separar qué explica la diferencia. Puede pesar el deflactor (IARAF publicó antes de
conocerse el IPC de abril y supuso 2,5%), qué conceptos entran en cada cifra
(transferencias totales o solo coparticipación) y cuándo registra Rentas cada
transferencia. Lo que sí muestra este contraste es que el escenario de −6,3% es, si
algo, conservador.

## 4. Ejemplos de uso: escenarios de caída real de 5%, 10% y 15%

Usando la misma `recaudacion_base` (masa coparticipable estimada para un cuatrimestre),
¿cómo cambiaría el impacto sobre Córdoba ante distintas magnitudes de shock?

In [6]:
escenarios = [-0.05, -0.10, -0.15]
filas = []
for variacion in escenarios:
    resultado = simular_shock(variacion, recaudacion_base_4m, COEFICIENTES_COPARTICIPACION)
    cordoba = [p for p in resultado["por_provincia"] if p["provincia"] == "Córdoba"][0]
    filas.append({
        "escenario": f"{variacion:.0%}",
        "impacto_total_provincias": resultado["total_impacto_provincias"],
        "impacto_cordoba": cordoba["impacto_pesos"],
    })

tabla_escenarios = pd.DataFrame(filas)
tabla_escenarios["impacto_total_provincias"] = tabla_escenarios["impacto_total_provincias"].map(lambda x: f"${x/1e9:,.1f} mil M")
tabla_escenarios["impacto_cordoba"] = tabla_escenarios["impacto_cordoba"].map(lambda x: f"${x/1e9:,.1f} mil M")
tabla_escenarios

,escenario,impacto_total_provincias,impacto_cordoba
0,-5%,"$-1,108.5 mil M",$-95.1 mil M
1,-10%,"$-2,216.9 mil M",$-190.1 mil M
2,-15%,"$-3,325.4 mil M",$-285.2 mil M


## 5. Simulador interactivo

Deslizá para elegir una caída (o suba) real de recaudación y ver el impacto resultante
en Córdoba y en el total de provincias, en vivo.

In [7]:
import ipywidgets as widgets
from IPython.display import display

slider = widgets.FloatSlider(
    value=round(variacion_implicita * 100, 1), min=-30, max=30, step=0.1,
    description="Variación %:", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="500px"),
)
salida = widgets.Output()


def actualizar(change):
    with salida:
        salida.clear_output(wait=True)
        variacion = slider.value / 100
        resultado = simular_shock(variacion, recaudacion_base_4m, COEFICIENTES_COPARTICIPACION)
        cordoba = [p for p in resultado["por_provincia"] if p["provincia"] == "Córdoba"][0]

        print(f"Variación de recaudación coparticipable: {variacion:+.1%}")
        print(f"Impacto total en las 24 provincias: ${resultado['total_impacto_provincias']/1e9:,.1f} mil millones")
        print(f"Impacto en Córdoba: ${cordoba['impacto_pesos']/1e9:,.1f} mil millones")

        fig, ax = plt.subplots(figsize=(6, 3.2), dpi=120)
        color = COLOR_NEGATIVO if cordoba["impacto_pesos"] < 0 else COLOR_POSITIVO
        ax.barh(["Córdoba"], [cordoba["impacto_pesos"] / 1e9], color=color, height=0.5)
        ax.axvline(0, color=GRID_COLOR, linewidth=1)
        ax.set_xlabel("Miles de millones de pesos")
        for spine in ["top", "right", "left"]:
            ax.spines[spine].set_visible(False)
        ax.tick_params(axis="both", length=0)
        fig.tight_layout()
        plt.show()


slider.observe(actualizar, names="value")
display(slider, salida)
actualizar(None)

FloatSlider(value=-6.3, continuous_update=False, description='Variación %:', layout=Layout(width='500px'), max…

Output()

## 6. Conclusiones

- El simulador (`scripts/simulator.py`) es una función pura, documentada y con 9 tests
  automáticos (`scripts/test_simulator.py`), que reparte un shock de recaudación entre
  provincias según los coeficientes de la Ley 23.548. Es deliberadamente simple: lineal
  y proporcional.
- Lo calibré con la caída de recaudación de comienzos de 2026 que reportó IARAF, con la
  masa de referencia llevada a precios de ese cuatrimestre. La variación implícita es
  ~−6,3%. Contra lo que Córdoba efectivamente cobró en ese cuatrimestre (−7,3% real,
  −\$134.100 millones), el modelo acierta el orden de magnitud y subestima la pérdida en
  alrededor de 10%.
- Para un shock de esa magnitud, Córdoba pierde del orden de \$120.100 millones de
  coparticipación en un cuatrimestre. En porcentaje, pierde lo mismo que cualquier otra
  provincia; la diferencia en vulnerabilidad la da la dependencia de cada una (Núcleo 4).